In [65]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [66]:
# Đọc file gốc
df = pd.read_csv("../Data/final_clean_data.csv")

# Chọn các cột cần thiết cho classification
selected_columns = [
    "MONTH","DAY_OF_MONTH","DAY_OF_WEEK",
    "OP_UNIQUE_CARRIER","ORIGIN","DEST",
    "CRS_DEP_TIME","CRS_ARR_TIME",
    "CRS_ELAPSED_TIME","DISTANCE",
    "HourlyDewPointTemperature","HourlyDryBulbTemperature",
    "HourlyRelativeHumidity","HourlyVisibility","HourlyWindSpeed",
    "DEP_DEL15"
]

df_classifi = df[selected_columns].copy()
print("Shape:", df_classifi.shape)
df_classifi.head()

Shape: (40486, 16)


,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,HourlyDewPointTemperature,HourlyDryBulbTemperature,HourlyRelativeHumidity,HourlyVisibility,HourlyWindSpeed,DEP_DEL15
0,4,1,2,AA,GEG,PHX,6.53,9.33,168.0,1020.0,33.0,36.0,89.0,5.0,10.0,0.0
1,4,1,2,AA,GEG,PHX,15.45,18.12,160.0,1020.0,33.0,45.0,63.0,10.0,10.0,1.0
2,4,1,2,AA,SEA,CLT,7.33,15.32,299.0,2279.0,37.0,43.0,80.0,10.0,6.0,0.0
3,4,1,2,AA,SEA,CLT,13.83,21.83,300.0,2279.0,41.0,49.0,74.0,10.0,6.0,0.0
4,4,1,2,AA,SEA,CLT,22.00,6.00,300.0,2279.0,41.0,45.0,86.0,10.0,7.0,0.0


In [67]:
# Cell 2: Missing value check
print("Số giá trị thiếu mỗi cột:\n", df_classifi.isnull().sum())

# Loại bỏ hàng có missing
df_classifi = df_classifi.dropna()
print("Sau khi loại missing:", df_classifi.shape)


Số giá trị thiếu mỗi cột:
 MONTH                        0
DAY_OF_MONTH                 0
DAY_OF_WEEK                  0
OP_UNIQUE_CARRIER            0
ORIGIN                       0
DEST                         0
CRS_DEP_TIME                 0
CRS_ARR_TIME                 0
CRS_ELAPSED_TIME             0
DISTANCE                     0
HourlyDewPointTemperature    0
HourlyDryBulbTemperature     0
HourlyRelativeHumidity       0
HourlyVisibility             0
HourlyWindSpeed              0
DEP_DEL15                    0
dtype: int64
Sau khi loại missing: (40486, 16)


In [68]:
# Cell 3: Check target distribution
print("Tỉ lệ phần trăm:\n", df_classifi["DEP_DEL15"].value_counts(normalize=True))


Tỉ lệ phần trăm:
 DEP_DEL15
0.0    0.872944
1.0    0.127056
Name: proportion, dtype: float64


In [69]:
# Cell 4: Encode categorical variables
label_cols = ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST"]
le = LabelEncoder()

for col in label_cols:
    df_classifi[col + "_ENC"] = le.fit_transform(df_classifi[col])

df_classifi = df_classifi.drop(columns=label_cols)
print("✅ Đã label encode các biến phân loại.")
df_classifi.head()


✅ Đã label encode các biến phân loại.


,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,HourlyDewPointTemperature,HourlyDryBulbTemperature,HourlyRelativeHumidity,HourlyVisibility,HourlyWindSpeed,DEP_DEL15,OP_UNIQUE_CARRIER_ENC,ORIGIN_ENC,DEST_ENC
0,4,1,2,6.53,9.33,168.0,1020.0,33.0,36.0,89.0,5.0,10.0,0.0,0,1,62
1,4,1,2,15.45,18.12,160.0,1020.0,33.0,45.0,63.0,10.0,10.0,1.0,0,1,62
2,4,1,2,7.33,15.32,299.0,2279.0,37.0,43.0,80.0,10.0,6.0,0.0,0,4,14
3,4,1,2,13.83,21.83,300.0,2279.0,41.0,49.0,74.0,10.0,6.0,0.0,0,4,14
4,4,1,2,22.00,6.00,300.0,2279.0,41.0,45.0,86.0,10.0,7.0,0.0,0,4,14


In [70]:
# Cell 5: Feature Engineering - TIME_OF_DAY & TEMP_DIFF

# 1️⃣ Nhóm khung giờ khởi hành (CRS_DEP_TIME)
def time_of_day(hour):
    # CRS_DEP_TIME là dạng số thập phân (ví dụ 13.75 ≈ 13h45)
    if 5 <= hour < 9:
        return "morning_peak"
    elif 9 <= hour < 15:
        return "midday"
    elif 15 <= hour < 19:
        return "evening_peak"
    else:
        return "night"

df_classifi["TIME_OF_DAY"] = df_classifi["CRS_DEP_TIME"].apply(time_of_day)

# Encode TIME_OF_DAY
le_time = LabelEncoder()
df_classifi["TIME_OF_DAY_ENC"] = le_time.fit_transform(df_classifi["TIME_OF_DAY"])

# 2️⃣ Tạo đặc trưng chênh lệch nhiệt độ
df_classifi["TEMP_DIFF"] = (
    df_classifi["HourlyDryBulbTemperature"] - df_classifi["HourlyDewPointTemperature"]
)

# Xóa cột TIME_OF_DAY gốc (chữ)
df_classifi = df_classifi.drop(columns=["TIME_OF_DAY"])

print("✅ Đã thêm feature: TIME_OF_DAY_ENC, TEMP_DIFF")
df_classifi[["CRS_DEP_TIME", "TIME_OF_DAY_ENC", "TEMP_DIFF"]].head()


✅ Đã thêm feature: TIME_OF_DAY_ENC, TEMP_DIFF


,CRS_DEP_TIME,TIME_OF_DAY_ENC,TEMP_DIFF
0,6.53,2,3.0
1,15.45,0,12.0
2,7.33,2,6.0
3,13.83,1,8.0
4,22.00,3,4.0


In [71]:
# Cell 6: Ensure numeric data types
for col in df_classifi.columns:
    df_classifi[col] = pd.to_numeric(df_classifi[col], errors='coerce')

print("✅ Kiểm tra kiểu dữ liệu hoàn tất:")
print(df_classifi.dtypes)


✅ Kiểm tra kiểu dữ liệu hoàn tất:
MONTH                          int64
DAY_OF_MONTH                   int64
DAY_OF_WEEK                    int64
CRS_DEP_TIME                 float64
CRS_ARR_TIME                 float64
CRS_ELAPSED_TIME             float64
DISTANCE                     float64
HourlyDewPointTemperature    float64
HourlyDryBulbTemperature     float64
HourlyRelativeHumidity       float64
HourlyVisibility             float64
HourlyWindSpeed              float64
DEP_DEL15                    float64
OP_UNIQUE_CARRIER_ENC          int64
ORIGIN_ENC                     int64
DEST_ENC                       int64
TIME_OF_DAY_ENC                int64
TEMP_DIFF                    float64
dtype: object


In [72]:
# Cell 7: Save to CSV for modeling
output_path = "../Data/final_classification_ready_v2.csv"
df_classifi.to_csv(output_path, index=False)

print(f"✅ File đã lưu: {output_path}")
print("Kích thước dữ liệu:", df_classifi.shape)


✅ File đã lưu: ../Data/final_classification_ready_v2.csv
Kích thước dữ liệu: (40486, 18)
